## What will you learn?

O que são modelos score driven para séries temporais. Como usar o pacote ScoerDrivenModels.jl de Julia para estimar e simular modelos GAS(p, q).

## What is a Score Driven Model?

Pegar da introdução do meu TCC.

Falar da intuição do score (taxa de variacao da verossimilhanca em relacao a um parametro da distribuicao. Quanto a verossimilhanca varia com a variacao da média?)

## What is a GAS(p, q) model?

Pegar da introdução do meu TCC. Falar que só serve para séries estacionarias por causa do processo ARMA(p, q). 

Falar que generalizar ARMA(p, q), GARCH(p, q), etc.

Falar que a estimação é via MLE e a simulação via Monte Carlo, com previsão fechada para 1 passo à frente.

## The ScoreDrivenModels.jl package

[Link para a documentação](https://lampspuc.github.io/ScoreDrivenModels.jl/latest/).

[Link para o artigo](https://arxiv.org/abs/2008.05506).

O artigo permite estimar modelos GAS(p,q) para séries temporais estacionarias de inúmeras distribuições, com 1 ou mais parâmetros variantes no tempo e para diferentes escalas (valores do hiperparâmetro *d*).

A tabela abaixo resume as distribuições com suas respectivas escalas implementadas.

Somado a isso, o usuário também pode adicionar uma nova distribuição ao pacote por meio de pull-requests. Veja este [link](https://lampspuc.github.io/ScoreDrivenModels.jl/latest/manual/#Implementing-a-new-distribution-1)

O pacote também permite diferentes otimizadores: 

Importante ressaltar que, como esses otimizadores são irrestritos e várias distribuições possuem restrições nos seus parâmetros (a variância não pode ser negativa, por exemplo), o pacote implementa [funções de ligação](https://lampspuc.github.io/ScoreDrivenModels.jl/latest/manual/#Recursion-1) para estes parâmetros.


## Let's see it in action

First of all, we need the following packages:



In [54]:
import Pkg
path = pwd()*"/TimeSeriesJulia/ScoreDrivenModels/"
Pkg.activate(path)
Pkg.instantiate()

  Activating project at `c:\Users\matno\OneDrive\Documentos\Diversos\MediumArticles\MathNogMediumArticles\TimeSeriesJulia\ScoreDrivenModels\TimeSeriesJulia\ScoreDrivenModels`


In [55]:
# Pkg.add("Dates")
# Pkg.add("Plots")
# Pkg.add("DelimitedFiles")
# Pkg.add("Distributions")
# Pkg.add("ScoreDrivenModels")

using Dates
using Plots
using DelimitedFiles
using Distributions
using ScoreDrivenModels

### Loading a time series

This is a monthly time series of the natural inflow energy of the Northeast region of Brazil, available inside the package.

I have dowloaded it from the package repo and saved it in the `data` folder in my [GitHub repository](https://github.com/MathNog/MathNogMediumArticles).

In [69]:
dates = collect(Date(1961):Month(1):Date(2000, 12));
y = vec(readdlm("data/nie_northeastern.csv"));

H  = 60
T  = 240
last_obs  = length(y) - H
first_obs = last_obs - T

y_train     = y[first_obs:last_obs];
y_test      = y[last_obs+1:end];

dates_train = dates[first_obs:last_obs];
dates_test  = dates[last_obs+1:end];

In [70]:
plot(dates_train, y_train, label = "Train")
plot!(dates_test, y_test, label = "Test") 
plot!(title="Brazil Northeastern Natural Inflow Energy", xlabel="Date", ylabel="Value")
plot!(xformatter = x -> Dates.format(Date(Dates.UTD(x)), "yyyy"))
savefig("output/northeastern_ts.png");

### Defining the model

We define the model using the `ScoreDrivenModel` function from the `ScoreDrivenModels` package. The function takes the following arguments:

- `p_lags`: The lags of the autoregressive part of the model.
- `q_lags`: The lags of the moving average part of the model.
- `dist`: The distribution of the model.
- `d`: The degree of the model.
- `time_varying_params`: The parameters of the model that are time-varying.


This is a crucial part of the modeling process, as it is when, based on the time-series features, we define the model's structure and the parameters that we want to estimate.

Since our time series is of monthly frequency and it is of a time series of natural inflow energy, we expect to have a strong seasonal pattern (as shown in the previous plot). Because of that, we will add a seasonal lag of 12 months in both the autoregressive and moving average parts of the model.

Moreover, since natural inflow can never be negative, it is natural to use a distribution that is always positive, such as the log-normal distribution. It is important to say that there is no technical reason not to use a Normal distribution. The model would be fitted and simulated without any issues, but we would possibly generate negative scenarios, which would not make sense in this context!

Finally, we will fit a Log-normal GAS model with both parameters as time varying, that is, both will follow an ARMA process with autoregressive and moving average lags of 1 and 12 months. 

In [ ]:
p_lags  = [1, 12]
q_lags  = [1, 12]
distrib = Distributions.LogNormal
d       = 0.0

gas = ScoreDrivenModel(p_lags, q_lags, distrib, d; time_varying_params = [1, 2]);

### Estimating and Simulating the model

In [72]:
fit_gas = fit!(gas, y_train);
forecast_gas = forecast(y_train, gas, test_months; S=1000);

Round 1 of 3: log-likelihood = -512.6292700958713
Round 2 of 3: log-likelihood = -514.3058384074782


┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matno\.julia\packages\Optim\7krni\src\types.jl:120
┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matno\.julia\packages\Optim\7krni\src\types.jl:120
┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (1.0e-6) will be used as f_reltol.
└ @ Optim C:\Users\matno\.julia\packages\Optim\7krni\src\types.jl:120


Round 3 of 3: log-likelihood = -510.28640261156556


### Visualize the fitted values and the simulated scenarios

In [77]:
point_forecast = forecast_gas.observation_forecast;
scenarios      = forecast_gas.observation_scenarios;
quantiles_obs  = forecast_gas.observation_quantiles;
fitted_values  = fitted_mean(gas, y_train);

In [74]:
plot(dates_train, y_train, label = "Train Values", color = :black)
plot!(dates_train, fitted_values, label = "Fitted Values", color = :blue)
plot!(dates_test, scenarios, label = "", color = :lightgray)
plot!(dates_test, y_test, label = "Test Values", color = :black)
plot!(dates_test, point_forecast, label = "Point Forecast", color = :red)
title!("Natural Inflow Energy Simulation from LogNormal GAS")
plot!(xformatter = x -> Dates.format(Date(Dates.UTD(x)), "yyyy"),
    legendcolumns = 4, legend = :outerbottom)
savefig("output/gas_simulation.png")

"c:\\Users\\matno\\OneDrive\\Documentos\\Diversos\\MediumArticles\\MathNogMediumArticles\\TimeSeriesJulia\\ScoreDrivenModels\\output\\gas_simulation.png"

### Visualize the fitted parameters and the simulated scenarios for each parameter

In [87]:
params_fitted = score_driven_recursion(gas, y_train)
params_point_forecast = forecast_gas.parameter_forecast; #H x P
params_scenarios      = forecast_gas.parameter_scenarios; #H x P x S

In [90]:
for i in 1:size(params_fitted, 2)
	plot(dates_train, params_fitted[2:end, i], label = "Fitted Parameters", color = :blue)
	plot!(dates_test, params_scenarios[:, i, :], label = "", color = :lightgray)
	plot!(dates_test, params_point_forecast[:, i], label = "Point Forecast", color = :red)
    title!("Parameter $i for LogNormal GAS")
    savefig("output/parameter_$i.png")
end